# Nonlinear Observation Processes

Reproduces Table `tab:app-nonlinear` and the family means quoted in the text (Appendix, Sensitivity Analyses). Fixed random 2-layer MLPs are applied to the data-generating process under three conditions; the linear condition reuses the main-experiment baseline regime.

| | Linear confounding | Nonlinear confounding |
|---|---|---|
| **Linear observations** | `linear` (main experiment) | `nonlinear_batch` |
| **Nonlinear observations** | `nonlinear_obs` | `nonlinear_both` |

**Setup:** baseline regime, $N_{\text{eval}}=1000$, pure scenarios, 50 seeds × 7 models. DSSL is averaged over its four $\beta$ configurations.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


def fmt(v: float, nd: int = 3) -> str:
    """Format with round-half-up, matching the paper (exact ties such as 0.5975 occur)."""
    return str(Decimal(f'{v:.10f}').quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))

DATA_DIR = Path('../../data')
NL_DIR   = DATA_DIR / 'nonlinear_experiment/results'
LIN_DIR  = DATA_DIR / 'main_experiment/results'

PURE = ('s1_', 's2b_', 's3_', 's4h_', 's4r_')
GT   = {'s1_': 'scenario_1', 's2b_': 'scenario_2', 's3_': 'scenario_3',
        's4h_': 'scenario_4', 's4r_': 'scenario_4'}

def get_gt(cfg):
    for p, s in GT.items():
        if cfg.startswith(p): return s

def model_family(m):
    if m.startswith('cca'):  return 'CCA'
    if m.startswith('clip'): return 'CLIP'
    if m.startswith('jive'): return 'JIVE'
    return 'DSSL'

# Load nonlinear conditions
records = []
for f in sorted(NL_DIR.glob('*.json')):
    cond = f.stem.rsplit('_', 1)[0]
    for r in json.load(open(f)):
        if r['n_eval'] == 1000 and any(r['alpha_config_id'].startswith(p) for p in PURE):
            r['condition'] = cond
            records.append(r)

# Load linear baseline (baseline measurement regime only)
for f in sorted(LIN_DIR.glob('baseline_baseline_*.json')):
    for r in json.load(open(f)):
        if r['n_eval'] == 1000 and any(r['alpha_config_id'].startswith(p) for p in PURE):
            r['condition'] = 'linear'
            records.append(r)

df = pd.DataFrame(records)
df['ground_truth'] = df['alpha_config_id'].map(get_gt)
df['model_family'] = df['model'].map(model_family)

print(f'Total records: {len(df)}')
print(f'Per condition: {dict(df["condition"].value_counts())}')

In [ ]:
CONDITIONS = {
    'linear':          'Linear (baseline)',
    'nonlinear_obs':   'Nonlinear observations',
    'nonlinear_batch': 'Nonlinear confounding',
    'nonlinear_both':  'Both nonlinear',
}
FAMILIES  = ['CCA', 'CLIP', 'JIVE', 'DSSL']
SCENARIOS = {'scenario_1': 'S1', 'scenario_2': 'S2B', 'scenario_3': 'S3', 'scenario_4': 'S4'}

rows = []
for fam in FAMILIES:
    for cond, cond_label in CONDITIONS.items():
        sub = df[(df['condition'] == cond) & (df['model_family'] == fam)]
        row = {'model': fam, 'condition': cond_label}
        for s, s_label in SCENARIOS.items():
            gt_sub = sub[sub['ground_truth'] == s]
            row[s_label] = (gt_sub['predicted_scenario'] == s).mean()
        non_s1 = sub[sub['ground_truth'] != 'scenario_1']
        row['FSCR'] = (non_s1['predicted_scenario'] == 'scenario_1').mean()
        rows.append(row)

full_table = pd.DataFrame(rows).set_index(['model', 'condition'])
print(full_table.to_string(float_format=fmt))

In [ ]:
# Family means quoted in the text: entangled = mean(CCA, CLIP), factorized = mean(JIVE, DSSL)
family_means = pd.concat({
    'Entangled':  full_table.loc[['CCA', 'CLIP']].groupby(level='condition', sort=False).mean(),
    'Factorized': full_table.loc[['JIVE', 'DSSL']].groupby(level='condition', sort=False).mean(),
})
print(family_means[['S2B', 'S3', 'S4', 'FSCR']].to_string(float_format=fmt))

In [ ]:
# Where S2B cases go when they are not detected (all models pooled)
s2b = df[df['ground_truth'] == 'scenario_2']
routing = (pd.crosstab(s2b['condition'], s2b['predicted_scenario'], normalize='index')
           .reindex(list(CONDITIONS))
           .rename(index=CONDITIONS))
print(routing.to_string(float_format=fmt))